In [1]:
# 从仓库根目录在新 kernel 中运行，使用当前仓库的 Henosis 源码。
import os
import sys
from pathlib import Path
from datetime import datetime
import json

organ='motc'
embed_org='motc'
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "Henosis" / "__init__.py").is_file():
    raise RuntimeError("请从包含 Henosis/ 的仓库根目录启动 notebook。")
DATA_ROOT = Path(
    os.environ.get("HENOSIS_DATA_DIR", PROJECT_ROOT / "data")
).expanduser()
DATA_DIR = f"{DATA_ROOT}/figure6/brain"

SEED = 449640641
GPU = "3"  # 按当前空闲情况修改；本次验证使用同型号 4090。
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    if os.environ.get("CUBLAS_WORKSPACE_CONFIG") != ":4096:8":
        raise RuntimeError("请先重启 kernel：CUBLAS_WORKSPACE_CONFIG 必须在 CUDA 初始化前设置。")
for name, module in tuple(sys.modules.items()):
    if name == "Henosis" or name.startswith("Henosis."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(PROJECT_ROOT / "Henosis"):
            raise RuntimeError("请先重启 kernel，避免混用其他目录与当前仓库的 Henosis 模块。")
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ[key] = "4"
os.environ["NUMBA_NUM_THREADS"] = "1"
sys.path.insert(0, str(PROJECT_ROOT))

import torch
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

RUN_DIR = PROJECT_ROOT / "saved_good_runs" / ("mosp_deterministic_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "config.json").write_text(json.dumps(dict(seed=SEED, gpu=GPU, torch=str(torch.__version__),
    cuda=torch.version.cuda, source=str(PROJECT_ROOT), deterministic=True), indent=2))
print("本次输出：", RUN_DIR)


本次输出： /mnt/public5/panguanyu/cross_species/scripts/saved_good_runs/mosp_deterministic_20261027_105141_371618


In [2]:
import os


import scanpy as sc
import seaborn as sns
from tqdm import tqdm
import scanpy.external as sce

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from pytorch_metric_learning import miners, losses

from Henosis.dataprocess.h5a import H5ADTokenizer
from Henosis.const import set_species_context
from Henosis import data
import Henosis.resa as resa
import Henosis.utils as utils
import Henosis.mmd as mmd
import Henosis.lr_schedule as lr_sche
import Henosis.swav as swav
import Henosis.peak_student as peak_student
import Henosis.henosis as henosis
#torch.manual_seed(16818030105712371645)
import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED % (2**32))
torch.manual_seed(SEED)  # Torch 保留完整 64 位 seed。


In [3]:
species_list=[
    'homo_sapiens_M1donor_rna','homo_sapiens_M1donor2_rna',
    'Macaca_mulatta_M1Dheni_atac','Macaca_mulatta_M1mac3_atac',
]

In [4]:
species_to_adata={
    k:sc.read_h5ad(f'{DATA_DIR}/h5ad/{k}.h5ad') for k in species_list
}

In [5]:
species_to_embed_file={
    'homo_sapiens_M1donor_rna':f"{DATA_DIR}/embedding/atac_ntv3_pre/homo_sapiens_M1donor_rna.avg.embeddings",
    'homo_sapiens_M1donor2_rna':f"{DATA_DIR}/embedding/atac_ntv3_pre/homo_sapiens_M1donor2_rna.avg.embeddings",
    'Macaca_mulatta_M1Dheni_atac':f"{DATA_DIR}/embedding/atac_ntv3_pre/Macaca_mulatta_M1Dheni_atac.avg.embeddings",
    'Macaca_mulatta_M1mac3_atac':f"{DATA_DIR}/embedding/atac_ntv3_pre/Macaca_mulatta_M1mac3_atac.avg.embeddings"
}

In [6]:
species_dict=set_species_context(species_to_adata)
token_species_dict={v: k for k, v in species_dict.items()}

In [8]:
tokenized = H5ADTokenizer(
    species_to_adata,
    species_to_embed_file,
).process_species_adata()
species_to_reps = {
    "homo_sapiens_M1donor_rna": "X_pca",
    "homo_sapiens_M1donor2_rna":'X_pca',
    "Macaca_mulatta_M1Dheni_atac": "X_lsi",
    "Macaca_mulatta_M1mac3_atac":"X_lsi"
}
species_cell_columns={
    "homo_sapiens_M1donor_rna": "Celltype",
    "homo_sapiens_M1donor2_rna":'Celltype',
    "Macaca_mulatta_M1Dheni_atac": "Celltype",
    "Macaca_mulatta_M1mac3_atac":"Celltype"
}
train_loader, test_loader = data.prepare_multispecies_dataloaders(
    tokenized,
    batch_size=256,
    species_to_reps=species_to_reps,
    batch_species=len(species_list),
    species_cell_columns=species_cell_columns,
    collate_fn=data.h5labelcollate_fn,
    num_workers=0,
)

100%|██████████| 9160/9160 [00:00<00:00, 16501.57it/s]


homo_sapiens_M1donor_rna: X_pca
homo_sapiens_M1donor2_rna: X_pca
Macaca_mulatta_M1Dheni_atac: X_lsi
Macaca_mulatta_M1mac3_atac: X_lsi


In [9]:
embed_size=1536
d_model=120
latent_size=64
num_layers=1
#net=peak_student.BtacDDPSwAV(species_to_embed_file,d_model,embed_size,latent_size,num_layers,10000,1,modality="atac",species_mapping=species_dict)
net=henosis.Henosis(species_to_embed_file,d_model,embed_size,latent_size,encoder_type='set',species_mapping=species_dict)

PCA ing
PCA done


In [10]:
def train_epoch(net,train_loader,loss_fn,loss_fn1,optimizer,scaler,device,current_temp,scheduler,ml_weight=1.0):
    net.train()
    metrics=utils.Accumulator(4)
    pbar=tqdm(train_loader,desc='Processing')
    for _,batch in enumerate(pbar):
        optimizer.zero_grad(set_to_none=True)

        input_ids,adata,sp,label=batch
        input_ids = input_ids.to(device, non_blocking=True)
        adata = adata.float().to(device, non_blocking=True)
        sp = sp.to(device, non_blocking=True)
        label=label.to(device, non_blocking=True)

        with autocast('cuda'):
            sl = torch.tensor(0.0, device=device)
            protein_features,gene_features,logit_scales=net(adata,input_ids,sp,)
            species_ids = torch.unique(sp).tolist()
            #for i in torch.unique(sp):
            for s in species_ids:
                mask = (sp == s)

                indices = torch.nonzero(mask).squeeze(1)
                gene_features_i = torch.index_select(gene_features, 0, indices)
                protein_features_i=torch.index_select(protein_features, 0, indices)

                sl += loss_fn(protein_features_i, gene_features_i,logit_scales[int(s)])
            
            #ml=loss_fn1(protein_features,sp)
            ml=loss_fn1(protein_features,sp)
            loss=sl+ml*ml_weight

        scaler.scale(loss).backward()
        if net.transformer.embed.gene_embed.weight.grad is not None:
            #net.transformer.embed.gene_embed.weight.grad[3:-1] = 0
            net.transformer.embed.gene_embed.weight.grad[4:] = 0
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if scheduler:
            scheduler.step()
        lr=optimizer.param_groups[0]['lr']
        pbar.set_description(f"loss:{loss.detach().item():.4f},sl:{sl.detach().item():.8f},ml:{ml.detach().item():.8f},lr:{lr:.8f}")
        metrics.add(loss.detach().item() * len(sp), sl.detach().item() * len(sp),ml.detach().item() * len(sp), len(sp))

    return {'loss':metrics[0] / metrics[3],'sl':metrics[1] / metrics[3],'ml':metrics[2] / metrics[3],}

def train(net,train_loader,test_loader,epochs,lr,species_to_h5,net_path,temp=0.2,ml_weight=1.0,warm_epoch=2):
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv1d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    (RUN_DIR / "training_config.json").write_text(json.dumps({
        "epochs": epochs, "lr": lr, "temp": temp,
        "ml_weight": ml_weight, "warm_epoch": warm_epoch,
    }, indent=2))
    torch.save(net.state_dict(), RUN_DIR / "initial_weights.pt")
    torch.save({"torch_cpu_rng": torch.get_rng_state(),
                "torch_cuda_rng": torch.cuda.get_rng_state_all(),
                "numpy_rng": np.random.get_state(), "python_rng": random.getstate()},
               RUN_DIR / "rng_before_training.pt")
    print('init done')

    device =torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('training on', device)
    net.to(device)

    loss_fn=swav.MultiScaleATACKNNLoss()#swav.SwAVLoss()#swav.WeightedMultiScaleATACKNNLoss()#swav.ATACKNNLoss()#
    loss_fn1=mmd.MMMD(kernel_type='rbf',each=True)#mmd.MSampleLoss(each=True)#
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr,fused=True)
    scheduler = lr_sche.CosineAnnealingWarmupRestarts(
        optimizer,
        first_cycle_steps=2000,   # 一个周期 2000 batch
        cycle_mult=1,             # 不增加周期长度
        max_lr=lr*1.5,            # 最大 lr 稍高
        min_lr=1e-6,
        warmup_steps=400,         # 前 400 batch 线性升到 max_lr
        gamma=0.6
    )
    early_stopping = utils.EarlyStopping(
        patience=10,               # 连续5个epoch指标不提升就停止
        verbose=True, 
        path=net_path, # 保存的文件名
        mode='min'                # 如果监控的是准确率，这里改用 'max'
    )
    scaler = GradScaler('cuda')
    cr_ml_weight=0.0
    try:
        for epoch in range(epochs):
            print(f"epoch:{epoch},")
            print(f"Temp: {temp}")
            if epoch>=warm_epoch:
                cr_ml_weight=ml_weight
            losses=train_epoch(net,train_loader,loss_fn,loss_fn1,optimizer,scaler,device,temp,scheduler,cr_ml_weight)
            current_lr = optimizer.param_groups[0]['lr']
            print(f"lr:{current_lr},")
            print(f'{losses}')
            for species in species_dict:
                print(species,net.logit_scales[species].cpu().detach().numpy())
            
            if epoch>=warm_epoch:
                early_stopping(losses['loss'],net)
                if early_stopping.early_stop:
                    print("====== Early stopping triggered! ======")
                    #infer_latent(net,test_loader,device,fcell_path,fsp_path,species_to_h5)
                    break
            if epoch ==epochs-1:
                torch.save(net.state_dict(), net_path)
                print(f'saving weight in {net_path}')
                #infer_latent(net,test_loader,device,fcell_path,fsp_path,species_to_h5)
                break
    except KeyboardInterrupt:
        #infer_latent(net,test_loader,device,fcell_path,fsp_path,species_to_h5)
        torch.save(net.state_dict(), net_path)
        print(f'saving weight in {net_path}')

In [11]:
net_path = str(RUN_DIR / "training_checkpoint.pt")
species_to_h5=tokenized
epochs=40
lr=1e-3
train(net,train_loader,test_loader,epochs,lr,species_to_h5,net_path,temp=0.0,ml_weight=1.0,warm_epoch=2)

init done
training on cuda
epoch:0,
Temp: 0.0


loss:13.1315,sl:13.13153934,ml:2.73127317,lr:0.00065681: 100%|██████████| 175/175 [00:15<00:00, 10.96it/s]


lr:0.0006568125000000001,
{'loss': 14.117144344874792, 'sl': 14.117144344874792, 'ml': 2.9195974567958287}
homo_sapiens_M1donor_rna 2.6741998
homo_sapiens_M1donor2_rna 2.6730714
Macaca_mulatta_M1Dheni_atac 2.6338618
Macaca_mulatta_M1mac3_atac 2.6832607
epoch:1,
Temp: 0.0


loss:12.1208,sl:12.12076378,ml:1.71269822,lr:0.00131262: 100%|██████████| 175/175 [00:14<00:00, 11.76it/s]


lr:0.001312625,
{'loss': 12.692479793003628, 'sl': 12.692479793003628, 'ml': 2.1824736097880773}
homo_sapiens_M1donor_rna 2.6893497
homo_sapiens_M1donor2_rna 2.7024093
Macaca_mulatta_M1Dheni_atac 2.551391
Macaca_mulatta_M1mac3_atac 2.7580285
epoch:2,
Temp: 0.0


...

lr:0.00030493028419225927,
{'loss': 10.882190181187221, 'sl': 10.729658126831055, 'ml': 0.1525320304291589}
homo_sapiens_M1donor_rna 3.321798
homo_sapiens_M1donor2_rna 3.37705
Macaca_mulatta_M1Dheni_atac 2.1913466
Macaca_mulatta_M1mac3_atac 2.7112427
EarlyStopping counter: 3 out of 10
epoch:38,
Temp: 0.0


loss:11.2886,sl:11.10606766,ml:0.18253860,lr:0.00027096: 100%|██████████| 175/175 [00:15<00:00, 11.30it/s]


lr:0.00027095677120779347,
{'loss': 10.821981811523438, 'sl': 10.670990486145019, 'ml': 0.1509913484113557}
homo_sapiens_M1donor_rna 3.322099
homo_sapiens_M1donor2_rna 3.3715498
Macaca_mulatta_M1Dheni_atac 2.1916995
Macaca_mulatta_M1mac3_atac 2.718499
Validation metric improved (10.855603 --> 10.821982). Saving model...
epoch:39,
Temp: 0.0


loss:11.1082,sl:10.94027901,ml:0.16792771,lr:0.00022430: 100%|██████████| 175/175 [00:15<00:00, 11.00it/s]

lr:0.00022430337432696198,
{'loss': 10.856445127214704, 'sl': 10.706587426321848, 'ml': 0.14985769131353924}
homo_sapiens_M1donor_rna 3.3180645
homo_sapiens_M1donor2_rna 3.3704252
Macaca_mulatta_M1Dheni_atac 2.1952748
Macaca_mulatta_M1mac3_atac 2.7224379
EarlyStopping counter: 1 out of 10
saving weight in /mnt/public5/panguanyu/cross_species/scripts/saved_good_runs/mosp_deterministic_20261027_105141_371618/training_checkpoint.pt


In [ ]:
from Henosis.evaluate import evaluate_integration
from threadpoolctl import threadpool_limits

evaluation_config = dict(
    metadata_columns=["Celltype"], embedding="protein", n_pcs=25,
    use_mot_correction=True, correction_by="species",
    correction_kwargs=dict(max_rounds=10, n_clusters=100, assignment_mode="sinkhorn", random_state=0),
    n_neighbors=15, umap_min_dist=0.3, umap_colors=["species", "Celltype"],
)
(RUN_DIR / "evaluation_config.json").write_text(json.dumps(evaluation_config, indent=2))
with threadpool_limits(limits=1):
    integrated_adata = evaluate_integration(net, test_loader, species_to_adata=tokenized, **evaluation_config)


infering: 100%|██████████| 118/118 [00:02<00:00, 56.73it/s]


KeyError: "homo_sapiens_M1donor_rna adata.obs missing columns: ['celltype']"

In [ ]:
# integrated_adata.write_h5ad(RUN_DIR / "integrated.h5ad")

In [ ]:
from collections import defaultdict
import pandas as pd
def attention_score_topk_freq(
    net,
    dataset,
    indices,
    device,
    species_to_embed_file,
    chosen_species,
    csv_path,
    top_k=10,
    top_n=10,
):
    net.eval()
    net.to(device)

    gene_freq = defaultdict(int)

    net.transformer.embed.gene_id(species_to_embed_file)
    raw_map = net.transformer.embed.species_gene_id_map[chosen_species]
    token_to_gene_name = {v: k for k, v in raw_map.items()}

    with torch.no_grad():
        for i in tqdm(indices):

            input_ids = dataset[i]['input_ids'].unsqueeze(0).to(device)
            sp = dataset[i]['sp'].unsqueeze(0).to(device)

            with autocast('cuda'):
                attn_score = net.transformer.attn_score(input_ids, sp)

            attn_score = attn_score.cpu().numpy()[0]   # [L, L]
            input_tokens = input_ids.cpu().numpy()[0]       # [L]

            # column importance: [L]
            cell_gene_importance = attn_score.mean(axis=0)

            # top-k indices
            topk_idx = np.argsort(cell_gene_importance)[-top_k:][::-1]

            for idx in topk_idx:
                token_id = input_tokens[idx]
                if token_id <= 3:
                    continue

                gene_name = token_to_gene_name.get(token_id, f"Unknown_{token_id}")
                gene_freq[gene_name] += 1

    # normalize to frequency
    n_cells = len(indices)
    final_scores = {
        gene: freq / n_cells
        for gene, freq in gene_freq.items()
    }

    sorted_genes = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)

    #top_n = 10
    '''print(f"\nTop {top_n} Genes by Top-{top_k} Attention Frequency:")

    top_genes=[]
    for rank, (gene, score) in enumerate(sorted_genes[:top_n]):
        print(f"Rank {rank+1}: {gene} (Freq: {score:.4f})")
        top_genes.append(gene)
    
    print(top_genes)'''
    
    df = pd.DataFrame(sorted_genes, columns=["gene", "score"])
    df.to_csv(csv_path, index=False)

    return sorted_genes[:top_n]

In [ ]:
'''import re
cell_type_column = 'celltype'
output_dir = RUN_DIR / 'tfc'
top_k = 1000
top_n = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(output_dir, exist_ok=True)

all_top_results = []
result_manifest = []

chosen_cell_types=['MGC','ASC','ODC']

for chosen_species, adata in tokenized.items():
    if cell_type_column not in adata.obs.columns:
        print(f'Skip {chosen_species}: obs 中没有 {cell_type_column!r} 列')
        continue

    x_rep = species_to_reps.get(chosen_species, 'X_pca')
    dataset = data.AnndataFastDataset(adata, chosen_species, X_rep=x_rep)
    cell_type_to_indices = defaultdict(list)
    for cell_index, cell_type in enumerate(adata.obs[cell_type_column].tolist()):
        if pd.notna(cell_type):
            cell_type_to_indices[str(cell_type)].append(cell_index)

    for cell_type, indices in tqdm(
        cell_type_to_indices.items(),
        total=len(cell_type_to_indices),
        desc=f'{chosen_species} cell types',
    ):
        if cell_type not in chosen_cell_types:
            continue

        safe_cell_type = re.sub(r'[^0-9A-Za-z._-]+', '_', cell_type).strip('_')
        safe_cell_type = safe_cell_type or 'unnamed'
        csv_path = os.path.join(
            output_dir,
            f'{organ}_{chosen_species}_{safe_cell_type}.csv',
        )

        top_results = attention_score_topk_freq(
            net,
            dataset,
            indices,
            device,
            species_to_embed_file,
            chosen_species,
            csv_path,
            top_k=top_k,
            top_n=top_n,
        )

        result_manifest.append({
            'species': chosen_species,
            'cell_type': cell_type,
            'n_cells': len(indices),
            'csv_path': csv_path,
        })
        all_top_results.extend(
            {
                'species': chosen_species,
                'cell_type': cell_type,
                'rank': rank,
                'gene': gene,
                'score': score,
                'n_cells': len(indices),
            }
            for rank, (gene, score) in enumerate(top_results, start=1)
        )

manifest_path = os.path.join(output_dir, f'{organ}_attention_score_topk_freq_manifest.csv')
summary_path = os.path.join(output_dir, f'{organ}_attention_score_topk_freq_top{top_n}.csv')
pd.DataFrame(result_manifest).to_csv(manifest_path, index=False)
pd.DataFrame(all_top_results).to_csv(summary_path, index=False)
print(f'完成 {len(result_manifest)} 个物种-细胞类型组合')
print(f'清单已保存到: {manifest_path}')
print(f'Top-{top_n} 汇总已保存到: {summary_path}')
pd.DataFrame(result_manifest)'''

In [ ]:
#pd.DataFrame(result_manifest)['csv_path'].tolist()

In [ ]:
#result_manifest